# AmazonHelp AI Support Agent

**Hiver SDE Intern take-home** — built on the Twitter Customer Support dataset (`twcs.csv`, ~2.8M tweets).

**Scope:** one brand, **@AmazonHelp** (170k support tweets, 21k reconstructable conversations). The agent must, for every incoming customer message:

1. **Classify** it into an intent taxonomy derived from the data (11 intents),
2. **Draft a reply** grounded in how AmazonHelp historically resolved similar issues,
3. **Decide** auto-handle vs. escalate-to-human, with a stated reason.

Everything runs **fully locally** (no paid APIs): `gemma4:e4b` drafts replies and classifies intent, `qwen3.5:4b` (a different model family — see decision log) judges reply quality.

> **How to read this notebook.** Data prep and the 3 non-LLM baselines re-run live in seconds. LLM stages are expensive (~30 min wall-clock on a laptop GPU), so their outputs are **committed to `results/`** by `run_pipeline.py` and loaded here. `python run_pipeline.py --fresh` regenerates everything from scratch.

In [ ]:
import sys, json, re
sys.path.insert(0, "src")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from agent import TAXONOMY, INTENTS, KEYWORD_RULES, clean_text, looks_english
pd.set_option("display.width", 160)

threads = pd.read_csv("data/amazonhelp_threads.csv")   # reconstructed conversations
golden  = pd.read_csv("golden_set.csv")                # hand-labelled eval set
summary = json.load(open("results/summary.json")) if __import__("os").path.exists("results/summary.json") else None
print(f"{threads.conv_id.nunique():,} conversations | {len(threads):,} tweets | {len(golden)} golden examples")

## 1. From 2.8M tweets to one brand's support conversations

`twcs.csv` is a flat edge list (`in_response_to_tweet_id`, `response_tweet_id`). I reconstructed conversation threads by walking reply chains: start from a customer tweet, follow the earliest reply edge at each hop (brands answer once per turn), cap depth at 30, and require the alternation customer → brand → customer → …

Brand selection over the 6 largest support accounts:

In [ ]:
brand_stats = pd.DataFrame([
    # (brand, outbound tweets, reconstructable convs, thread tweets, median turns)
    ("AppleSupport",  106_860, 49_261, 132_999, 2),
    ("AmazonHelp",    169_840, 21_423,  78_686, 3),
    ("Delta",          42_253, 24_134,  63_801, 2),
    ("Uber_Support",   56_270, 18_877,  50_501, 2),
    ("Tesco",          38_573, 14_769,  42_421, 2),
    ("SpotifyCares",   43_265, 12_687,  35_298, 2),
], columns=["brand", "outbound", "convs", "thread_tweets", "median_turns"])
brand_stats["tweets_per_conv"] = (brand_stats.thread_tweets / brand_stats.convs).round(1)
brand_stats

In [ ]:
# conversations look like this (one example, trimmed)
demo = threads[threads.conv_id == threads.conv_id.iloc[0]]
for _, r in demo.head(6).iterrows():
    who = "CUSTOMER" if r.inbound else "  BRAND  "
    print(f"{who} d{int(r.depth)}: {clean_text(r.text)[:95]}")

**Why AmazonHelp:** AppleSupport has more conversations, but its median thread is **2 turns** — one deflection ("DM us") and done. AmazonHelp is the only top brand with **median 3 turns** (real back-and-forth: 78,686 tweets over 21,423 conversations, 254 distinct agents signing `^XX` on 96.7% of its 36,411 in-thread replies), and its reply mix includes genuine resolution content (refund guidance, delivery instructions, links) alongside deflection. Real resolution content is what a reply-drafting agent can learn from. Full numbers: `report/brand_selection.md`.

**What "good" means here** (used for the judge rubric and route design):
- *Grounded* — addresses the actual issue, not a generic apology
- *Safe* — never asks for order numbers / passwords publicly, never promises money publicly, stays inside AmazonHelp's historical behaviour (privacy-deflection to DM)
- *Helpful* — gives a concrete next step
- *On-brand* — brief, warm, ends with an agent sigil `^XX`

## 2. Intent taxonomy, derived from the data

K-means on TF-IDF collapsed into noise (5,986/6,000 messages in one cluster — sparse multilingual text), so I derived the taxonomy bottom-up instead: (a) read hundreds of messages across thread depths, (b) bucket coverage stats for candidate keyword sets, (c) name the buckets that actually appear. Result: **11 intents** (10 + OTHER).

In [ ]:
tax = pd.DataFrame({"intent": INTENTS, "definition": [TAXONOMY[i] for i in INTENTS]})
tax

In [ ]:
# keyword-rule coverage over the English training pool — the labelling rule for weak supervision
firsts = (threads.sort_values("depth").groupby("conv_id").first()
          .query("inbound == True").reset_index())
replies = threads.query("inbound == False and depth == 1")[["conv_id", "text"]]
pairs = firsts[["conv_id", "text"]].merge(replies, on="conv_id", suffixes=("_msg", "_reply"))
pairs.columns = ["conv_id", "msg", "reply"]
pool = pairs[~pairs.conv_id.isin(set(golden.conv_id))]
pool = pool[pool.msg.map(looks_english)]
cov = pool.msg.map(lambda t: next((i for i, p in KEYWORD_RULES.items()
                                   if re.search(p, clean_text(t), re.I)), None))
print(f"English pool: {len(pool):,} | keyword-labelled: {cov.notna().sum():,} ({cov.notna().mean():.1%})")
print(cov.value_counts().to_string())

Two design consequences of these numbers:

- **Keyword coverage is 57.6%** of English messages → keyword rules are a legitimately weak *trivial baseline*, and the ~8k weak labels are a decent distant-supervision pool for the TF-IDF classifier.
- `OTHER` matters: 29/200 golden messages are product questions / off-topic / unclear — an agent that forces everything into a support intent looks robotic.

## 3. Golden evaluation set — 200 hand-labelled examples

**Sampling:** `seed=42`, uniform random over the same pool the agent serves (first-inbound English messages of non-overlapping conversations). Weak keyword labels pre-filled the spreadsheet; **every example was then read and corrected by hand** (intent + route). Ten pre-labelled items that turned out to be Spanish/German/Italian/Portuguese despite passing an ASCII filter were swapped for English alternates (protocol in `report/golden_protocol.md`).

**Route labelling rubric** (fixed *before* looking at any system output):
- *human* — money/fraud/account/complaint involved, angry or profane, legal/churn threat, non-English, or issue needs account access
- *auto* — routine status/ETA/how-to questions that a safe template answer handles

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
golden.intent.value_counts().plot.barh(ax=ax[0], color="#1f77b4")
ax[0].set_title("Golden set: intent distribution (n=200)")
golden.route.value_counts().plot.bar(ax=ax[1], color=["#2ca02c", "#d62728"])
ax[1].set_title("Route: auto vs human")
plt.tight_layout(); plt.show()

print(pd.crosstab(golden.intent, golden.route, normalize="index").round(2).mul(100).astype(int)
      .rename(columns={0: "auto %", 1: "human %"}).to_string())

The crosstab is the empirical basis for the a-priori escalation policy: `SERVICE_COMPLAINT` (88% human), `ACCOUNT`, `REFUND`, `PAYMENT` are human-heavy lanes; `ORDER_STATUS` is mostly auto-able.

## 4. The agent (`src/agent.py`)

| Component | Choice | Why |
|---|---|---|
| Intent | `gemma4:e4b`, batched JSON classification | 10-way few-shot LLM beats weak-label-trained models (results below) |
| Reply | RAG: TF-IDF top-3 historical (message → AmazonHelp reply) pairs + `gemma4:e4b` rewrite | Grounded in real resolutions; prompt enforces ≤2 sentences, no public data requests, `^XX` sigil |
| Escalation | a-priori rule set, **not tuned on the golden set** | non-English OR conf<0.45 OR high-risk intent OR lexicon hit (profanity/anger/fraud/legal/churn/sensitive) OR money amount OR retrieval sim<0.12 |

Baselines (all evaluated on the same 200 examples):
1. **Trivial (intent):** majority class (`ORDER_STATUS`)
2. **Trivial (reply):** canned "please contact us here: <link>" — AmazonHelp's single most common real reply archetype
3. **Simple (intent):** TF-IDF + LogisticRegression on ~8k keyword weak labels
4. **Simple (reply):** verbatim nearest-neighbour historical reply

In [ ]:
from agent import escalation_reasons
for t, i, c, s in [
    ("where is my order?? 3 days late", "ORDER_STATUS", 0.9, 0.4),          # auto
    ("I want a refund for the $60 you stole from me", "REFUND", 0.9, 0.4),  # human: money+refund
    ("your support is pathetic, I'm cancelling prime", "SERVICE_COMPLAINT", 0.9, 0.4),
]:
    print(f"[{'HUMAN' if escalation_reasons(t,i,c,s) else 'AUTO '}] {t[:55]:57s} -> {escalation_reasons(t,i,c,s)}")

## 5. Results

### 5.1 Intent classification

In [ ]:
preds = pd.read_csv("results/intent_preds.csv")
sys.path.insert(0, "."); from evaluate import intent_report
pd.DataFrame([
    intent_report(preds.gold, ["ORDER_STATUS"]*len(preds), "majority (trivial)"),
    intent_report(preds.gold, preds.kw, "keyword rules (trivial)"),
    intent_report(preds.gold, preds.ml, "tfidf+logreg (simple)"),
    intent_report(preds.gold, preds.llm, "LLM gemma4:e4b (agent)"),
]).round(3)

In [ ]:
# where the agent is wrong: top confusions
wrong = preds[preds.llm != preds.gold]
conf = wrong.groupby(["gold", "llm"]).size().sort_values(ascending=False).head(8)
print(conf.to_string())

### 5.2 Routing (auto vs. human)

The error that matters most is a **missed escalation**: a message that needed a human but was auto-answered. Route policy applied on top of each intent system's (intent, confidence):

In [ ]:
route_df = pd.read_csv("results/route_preds.csv")
from evaluate import route_metrics
rows = []
for system in ["keyword", "ml", "llm"]:
    m = route_metrics(preds.route_gold, route_df[route_df.system == system].route)
    rows.append({"system": system, **{k: v for k, v in m.items() if k != "confusion"}})
pd.DataFrame(rows).round(3)

### 5.3 Reply quality — LLM-as-judge

`qwen3.5:4b` scores every reply 1–5 on the four rubric dimensions defined in §1 (groundedness, helpfulness, safety/tone, brand voice). Different model family from the drafter to avoid self-preference bias. Judge evaluated all three reply systems on the same 50-message random subsample (seed 42).

In [ ]:
judge = pd.read_csv("results/judge_scores.csv")
jm = judge.groupby("system")[["groundedness", "helpfulness", "safety_tone", "brand_voice"]].mean()
jm["overall"] = jm.mean(axis=1)
jm.round(2).sort_values("overall", ascending=False)

### 5.4 Does the judge agree with a human?

I hand-rated all 50 agent drafts on the same rubric (`results/hand_ratings.csv`), then measured judge–human agreement:

In [ ]:
agree = pd.read_csv("results/judge_human_agreement.csv")
agree.round(3)

## 6. Failure analysis (top modes, real examples)

In [ ]:
# Mode 1 evidence: intent confusions with texts, plus missed escalations
wrong_txt = wrong.assign(text=wrong.ex_id.map(golden.set_index("ex_id").text))
for (g, p), n in wrong_txt.groupby(["gold", "llm"]).size().sort_values(ascending=False).head(4).items():
    ex = wrong_txt[(wrong_txt.gold == g) & (wrong_txt.llm == p)].iloc[0]
    print(f"[{n}x] gold={g} pred={p}\n  {clean_text(ex.text)[:120]}")
print("\n(full evidence with missed escalations + false alarms: report/failures.md)")

The five modes, counts and examples in `report/report.md` §4 (evidence in `report/failures.md`):
1. **Missed escalation: "delivered but not received" (11 of 28 misses)** — loss/fraud allegations predicted DELIVERY_ADDRESS / DAMAGED_OR_WRONG, neither in the high-risk intent set, so "standard flow" auto-answers them
2. **Obfuscated profanity evades the lexicon** — "f*&k$" style writes pass the regexes
3. **Public order numbers don't trigger privacy escalation** — ex 24 tweets a full order number; I tested a `\d{3}-\d{7}-\d{7}` trigger: recovers 2 misses, creates 6 false alarms — a real tradeoff, not a free win
4. **Vague messages slip under the confidence floor** — "Help me please." gets conf 0.50 > 0.45 and auto-handles with nothing to ground a reply in
5. **Language detector false-alarms on English with jargon (14 of 39 flags)** — "Kids Edition HD8", "satna mp 485001": product/place names read as non-English

## 7. What is misleading about my headline number?

**Intent accuracy (75%) and macro-F1 overstate end-to-end quality** for four reasons:

1. **The golden set is first-turn only, English-only, clean-text.** Real @AmazonHelp traffic is multilingual (10 of my 200 random samples were non-English despite an ASCII pre-filter — 5%), multi-turn (context lives in prior turns), and full of spam/emoji noise. My agent answers turn-1 of a conversation it never saw the history of.
2. **Reply-quality numbers are judged, not measured.** `qwen3.5:4b` is a 4B model; its agreement with my own ratings (per-dimension Spearman and weighted kappa in §5.4) is the ceiling on how much those numbers can be trusted — every reply-quality figure inherits that noise.
3. **The route "accuracy" hides asymmetric costs.** A missed escalation (angry customer auto-answered) costs far more than a false alarm (human glances at a routine question); the single 0.66 number invites the wrong reading, the confusion counts matter more.
4. **No A/B against real humans.** The only ground truth for "would this reply have resolved the issue" is what AmazonHelp actually did — and the brand's own most common reply is a link-deflection, so "matching the brand" is itself a low bar that the judge rewards.

## 8. With one more week

1. Route on **required-capability, not topic** (fixes the 11 loss/fraud misses): ask "what would resolving this need — account lookup / order-system action / policy exception?" and escalate on that
2. Real **language ID** (fixes 14 false alarms): trained char-ngram model instead of common-word fraction; unlocks answering the Hindi/Portuguese lanes
3. **Calibrate, don't hand-pick**: sweep conf/similarity thresholds on held-out weak-labelled data; report the missed-escalation/false-alarm curve instead of the a-priori point
4. **Multi-turn drafting** (top scope cut): condition drafts on the full thread; the retrieval index already stores conversations
5. **Judge upgrade**: pairwise A/B judging (LLM judges rank far more reliably than they score absolutely) + a second human rater

## 9. Decision log (non-obvious calls)

| # | Decision | Why |
|---|---|---|
| 1 | AmazonHelp over AppleSupport | only top brand with median 3-turn conversations; Apple = single deflection template |
| 2 | 11-intent taxonomy from keyword buckets, not clustering | MiniBatchKMeans collapsed 99.8% of messages into one cluster (sparse multilingual TF-IDF) |
| 3 | Local Ollama models, no APIs | Zero-cost, reproducible offline; assignment permits any model |
| 4 | Drafter `gemma4:e4b`, judge `qwen3.5:4b` — different families | Avoids judge self-preference for its own family's style |
| 5 | `think: false` on qwen | Thinking models return empty `/api/chat` content otherwise |
| 6 | Weak labels (57.6% keyword coverage) as ML training signal, not gold | Cheap scale; keyword coverage itself justifies keyword-as-trivial-baseline |
| 7 | Escalation policy fixed a priori (0.45 / 0.12 / lexicon) | Tuning on the golden set would inflate route metrics dishonestly |
| 8 | Golden set excludes golden conv_ids from retrieval + ML pool | Leakage: a retrieved golden reply would make reply eval meaningless |
| 9 | 10 non-English golden items swapped for English alternates | The English-only agent can't be fairly scored on them; recorded in protocol |
| 10 | Committed LLM outputs in `results/` | 15-min reproduction budget; ~30-min fresh run documented in README |
| 11 | LLM batch=10 JSON classification | Cuts 200 messages to 20 calls; batch 20 overflowed the default num_ctx (raised to 8192) |
| 12 | Canned reply = AmazonHelp's modal real reply | Trivial baseline must be the brand's own most common answer, not a strawman |
| 13 | Judge on 50-message subsample, not 200 | Judge latency ~15s/reply; 3 systems × 200 would exceed the time budget |
| 14 | Route metrics separate missed-escalations from false alarms | Accuracy alone hides the asymmetric cost structure |

---

**Reproduce:** `pip install -r requirements.txt`, `ollama pull gemma4:e4b && ollama pull qwen3.5:4b`, then `python run_pipeline.py` (uses committed caches, ~3 min) or `python run_pipeline.py --fresh` (~30 min, 2 LLMs). Full details in `README.md` and `report/report.md`.